# Inference: Base, Teacher, Student

Runs all three models on 120 German customer-support queries and saves the raw responses.<br> 
These responses will be used to compare model performances in notebook 5<br>
Results are printed directly as notebook output. Optionally, they can be saved locally via the switch in Section 4.<br>
<br>
**Pipeline position:** 01 Fine-Tuning → 02 Data Generation → 03 Student Distillation → `[04 Inference]` → 05 Evaluation<br>
**Strong GPU required.** This notebook was developed on a Kaggle T4 (16 GB VRAM).<br>
[![Open Notebook in Kaggle](https://img.shields.io/badge/Open%20Notebook%20in-Kaggle-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/04-inference)

## 1. Setup

Install dependencies and check GPU.  
Local users: skip the pip cell — install via `pip install -r requirements.txt` instead.  
This notebook is inference-only — no training libraries needed.

In [1]:
%%capture
!pip install -q --upgrade unsloth transformers peft bitsandbytes accelerate python-dotenv

In [2]:
import gc
import json
import time
import os
import torch
from datetime import date


os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Kaggle's GPU T4 x2 session auto-splits the model across both GPUs, causing a device-mismatch crash; pin to one T4 to avoid it

print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"VRAM:             {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version:  2.10.0+cu128
CUDA available:   True
GPU:              Tesla T4
VRAM:             15.6 GB


### **Models evaluated in this notebook**

Three models are compared — loaded one at a time, in this order:

- **Base** — `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` (no fine-tuning).
  - The out-of-the-box 1B model. Serves as the baseline: shows what the student model looks like before distillation.
- **Teacher** — `LeoLM/leo-mistral-hessianai-7b-chat` with LoRA adapter `Feyerade/german-support-leollm-lora-adapter`.
  - The 7B model fine-tuned on synthetic German support data in Notebook 01. Sets the target style the student is trained to match.
- **Student** — `Feyerade/german-support-llama-1b-distilled`.
  - The 1B model distilled from teacher outputs in Notebook 03. Standalone merged checkpoint — no adapter needed.

In [3]:
BASE_MODEL_ID    = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
TEACHER_BASE_ID  = "LeoLM/leo-mistral-hessianai-7b-chat"
TEACHER_ADAPTER  = "Feyerade/german-support-leollm-lora-adapter"
STUDENT_MODEL_ID = "Feyerade/german-support-llama-1b-distilled"

print(f"Base model:      {BASE_MODEL_ID}")
print(f"Teacher base:    {TEACHER_BASE_ID}")
print(f"Teacher adapter: {TEACHER_ADAPTER}")
print(f"Student model:   {STUDENT_MODEL_ID}")

Base model:      unsloth/Llama-3.2-1B-Instruct-bnb-4bit
Teacher base:    LeoLM/leo-mistral-hessianai-7b-chat
Teacher adapter: Feyerade/german-support-leollm-lora-adapter
Student model:   Feyerade/german-support-llama-1b-distilled


## 2. Queries, Parameters, and Helpers

Defines the 120 evaluation queries, generation parameters, and the helper functions used in Section 4.

In [4]:
from collections import Counter

test_queries = [
    # ===== Orders & Shipping (24) =====
    # -- polite (3 simple / 3 ambiguous / 2 clear) --
    {"id": "q001", "query": "Guten Tag, ich wollte fragen ob Sie auch in die Schweiz liefern. Falls ja, wie hoch waeren die Versandkosten?", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q002", "query": "Koennten Sie mir sagen, bis zu welcher Uhrzeit ich bestellen muss, damit die Ware noch am selben Tag versendet wird?", "category": "Orders & Shipping", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q003", "query": "Ich wollte einfach mal ein Lob dalassen: meine letzte Lieferung kam blitzschnell und tadellos verpackt an. Machen Sie weiter so!", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q004", "query": "Ich moechte meine Bestellung an eine DHL-Packstation liefern lassen. Wo trage ich die Postnummer im Bestellprozess ein?", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q005", "query": "Koennten Sie mir zeigen, wo ich im Shop die voraussichtliche Lieferzeit eines Artikels sehe, bevor ich ihn in den Warenkorb lege?", "category": "Orders & Shipping", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q006", "query": "Bei meiner naechsten Bestellung wuerde ich gern ein Wunschlieferdatum angeben. Wo finde ich diese Option im Bestellablauf?", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q007", "query": "Ich habe drei Artikel bestellt und moechte zwei davon an meine Heimadresse und einen an mein Buero schicken lassen. Wie richte ich einen solchen geteilten Versand ein?", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    {"id": "q008", "query": "Ich bin umgezogen und moechte eine neue Standardlieferadresse hinterlegen und die alte entfernen. Koennten Sie mir die noetigen Schritte erklaeren?", "category": "Orders & Shipping", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    # -- frustrated (3 simple / 2 ambiguous / 3 clear) --
    {"id": "q009", "query": "Mein Paket hat seit Donnerstag den Status in Zustellung, aber bei mir kommt einfach nichts an.", "category": "Orders & Shipping", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q010", "query": "Ich habe extra acht Euro fuer Expressversand bezahlt und die Sendung kommt spaeter als der normale Standardversand. Was soll das.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q011", "query": "Beim Checkout standen zwei Tage Lieferzeit, nach der Bestellung sind es ploetzlich zehn Werktage. So gewinnt man keine Kunden.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q012", "query": "Ich will die Sendungsverfolgung zu meiner Bestellung aufrufen, finde den Link aber nirgends in meinem Konto. Wo versteckt der sich.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q013", "query": "Ich moechte fuer meine offene Bestellung den Zustelldienst wechseln, sehe dafuer aber keine Option. Wo ist die bitte.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q014", "query": "Der Bote hat mein Paket offen im Hausflur abgestellt, jetzt fehlen zwei Artikel aus der Sendung. Das ist dieses Jahr schon das dritte Mal. Ich erwarte eine Loesung.", "category": "Orders & Shipping", "sentence_count": 3, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q015", "query": "Ich moechte zwei Artikel aus meiner letzten Bestellung zuruecksenden, finde im Konto aber keine klare Anleitung fuer die Ruecksendung. Wie laeuft der Rueckversand Schritt fuer Schritt ab.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q016", "query": "Ich habe zwei getrennte Bestellungen aufgegeben und will, dass sie zusammen in einem Paket ankommen, um Verpackung zu sparen. Wie bekomme ich das organisiert.", "category": "Orders & Shipping", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    # -- concise (2 simple / 3 ambiguous / 3 clear) --
    {"id": "q017", "query": "Versand nach Belgien moeglich", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q018", "query": "Samstagszustellung ja oder nein", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q019", "query": "Sendungsverfolgung im Konto aufrufen wo", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q020", "query": "Versand nachtraeglich beschleunigen wie", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q021", "query": "Packstation als Lieferadresse eintragen wo", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q022", "query": "Lieferadresse aendern und alte entfernen Ablauf", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q023", "query": "Bestellung auf zwei Adressen aufteilen wie", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q024", "query": "Artikel zuruecksenden und Ruecksendeetikett erstellen Ablauf", "category": "Orders & Shipping", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    # ===== Returns & Complaints (24) — immer Rueckerstattung des Kaufbetrages =====
    # -- polite (3 simple / 3 ambiguous / 2 clear) --
    {"id": "q025", "query": "Falls ein Geraet innerhalb der Gewaehrleistung defekt wird, erhalte ich dann den Kaufbetrag zurueck oder nur einen Umtausch?", "category": "Returns & Complaints", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q026", "query": "Ich habe vor einem Jahr einen Akku-Staubsauger gekauft, dessen Akku stark nachgelassen hat. Habe ich Anspruch auf eine teilweise Erstattung des Kaufpreises?", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q027", "query": "Ich wollte mich nur kurz bedanken, meine letzte Rueckerstattung war schon nach zwei Tagen auf dem Konto. Das lief wirklich vorbildlich!", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q028", "query": "Koennten Sie mir sagen, wo ich in meinem Konto den Status einer angeforderten Rueckerstattung einsehen kann?", "category": "Returns & Complaints", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q029", "query": "Ich moechte fuer eine Erstattung eine andere Bankverbindung angeben. Wo trage ich diese in meinem Konto ein?", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q030", "query": "Wo sehe ich, auf welches Zahlungsmittel meine anstehende Erstattung gutgeschrieben wird?", "category": "Returns & Complaints", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q031", "query": "Mein gestern geliefertes Geraet ist defekt und ich moechte statt eines Ersatzes den vollen Kaufbetrag erstattet bekommen. Koennten Sie mir den Ablauf der Rueckerstattung erklaeren?", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    {"id": "q032", "query": "Ich habe eine dreiteilige Bestellung erhalten und moechte fuer zwei bereits zurueckgegebene Artikel die Erstattung des Kaufbetrages anstossen. Wie gehe ich dabei vor?", "category": "Returns & Complaints", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    # -- frustrated (3 simple / 2 ambiguous / 3 clear) --
    {"id": "q033", "query": "Ich habe einen defekten Toaster vor drei Wochen zurueckgeschickt, aber die Erstattung des Kaufbetrages ist bis heute nicht auf meinem Konto.", "category": "Returns & Complaints", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q034", "query": "Drei von vier gelieferten Weinglaesern waren angeschlagen, und fuer die zwei reklamierten Glaeser habe ich bis heute keinen Cent zurueckbekommen. Ist das normal bei euch.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q035", "query": "Mein Saugroboter ist nach fuenfzehn Monaten kaputt, ihr verweigert aber trotz beworbener vierundzwanzig Monate Garantie jede Erstattung. Was soll dieser Werbeschwindel.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q036", "query": "Ich will nachvollziehen, wie viel Geld ihr mir fuer meine Reklamation erstattet habt, finde in eurem Konto aber keine Aufstellung dazu. Wo steht das.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q037", "query": "Ich versuche fuer meine Erstattung die neue Kontonummer zu hinterlegen und das Feld dafuer laesst sich einfach nicht speichern. Wo trage ich das sonst ein.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q038", "query": "Die gelieferte Hose ist viel kleiner als angegeben, ich habe sie zurueckgeschickt und verlange jetzt neben dem Kaufpreis auch die Erstattung meiner Versandkosten. Wie bekomme ich beide Betraege zurueck.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q039", "query": "Mein Geraet wurde zweimal repariert und ist wieder defekt, jetzt will ich vom Kauf zuruecktreten und den vollen Kaufbetrag erstattet bekommen. Wie laeuft diese Rueckerstattung ab.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q040", "query": "Ihr habt mir das falsche Modell geliefert und ich will keinen Ersatz mehr, sondern den kompletten Kaufbetrag zurueck. Wie fordere ich die volle Erstattung an.", "category": "Returns & Complaints", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    # -- concise (2 simple / 3 ambiguous / 3 clear) --
    {"id": "q041", "query": "Erstattung auf andere Karte moeglich", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q042", "query": "Erstattung ohne Kaufbeleg moeglich", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q043", "query": "Status meiner Erstattung einsehen wo", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q044", "query": "Bankverbindung fuer Erstattung aendern wo", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q045", "query": "Erstattung als Gutschein statt auf Karte wie", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q046", "query": "Erstattung fuer zwei zurueckgegebene Artikel beantragen Ablauf", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q047", "query": "Kaufbetrag fuer defektes Geraet zurueckfordern Schritte", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q048", "query": "Vom Kauf zuruecktreten und vollen Kaufbetrag erstatten lassen wie", "category": "Returns & Complaints", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    # ===== Payment & Billing (24) — Abrechnung, Zahlungswege, Buchungsfehler =====
    # -- polite (3 simple / 3 ambiguous / 2 clear) --
    {"id": "q049", "query": "Bieten Sie fuer Kunden mit gueltigem Studentenausweis einen Rabatt an?", "category": "Payment & Billing", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q050", "query": "Ich wollte fragen, ob bei Ihnen auch die Zahlung per SEPA-Lastschrift moeglich ist oder nur per Kreditkarte.", "category": "Payment & Billing", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q051", "query": "Ich wollte nur einmal loben, dass Ihre Rechnungen so uebersichtlich und nachvollziehbar aufgebaut sind. Das ist heute leider selten!", "category": "Payment & Billing", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q052", "query": "Koennten Sie mir zeigen, wo ich in meinem Konto meine bisherigen Rechnungen herunterladen kann?", "category": "Payment & Billing", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q053", "query": "Ich moechte kuenftig per SEPA-Lastschrift statt per Kreditkarte zahlen. Wo hinterlege ich meine Bankverbindung im Konto?", "category": "Payment & Billing", "sentence_count": 2, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q054", "query": "Wo kann ich in meinem Konto einen Gutscheincode einloesen, den ich per Newsletter erhalten habe?", "category": "Payment & Billing", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q055", "query": "Ich moechte eine groessere Anschaffung ueber achthundert Euro in mehreren Monatsraten zahlen. Koennten Sie mir erklaeren, wie ich eine Ratenzahlung einrichte?", "category": "Payment & Billing", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    {"id": "q056", "query": "Ich benoetige fuer meine Buchhaltung kuenftig Rechnungen auf meine Firma statt auf meinen Namen. Wie stelle ich mein Konto entsprechend um?", "category": "Payment & Billing", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    # -- frustrated (3 simple / 2 ambiguous / 3 clear) --
    {"id": "q057", "query": "Ihr habt mein Konto trotz meiner schriftlichen Kuendigung diesen Monat schon wieder fuer das Abo belastet.", "category": "Payment & Billing", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q058", "query": "Der Rabattcode aus eurem Newsletter greift angeblich erst ab fuenfzig Euro Mindestbestellwert. Davon stand in der Mail kein Wort.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q059", "query": "Mein Konto wurde doppelt belastet, obwohl ich nur einmal bestellt habe. So geht man mit Kunden nicht um.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q060", "query": "Ich will meine hinterlegte Kreditkarte durch eine neue ersetzen, finde die Option dafuer aber nirgends. Wo versteckt ihr die.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q061", "query": "Ich suche seit einer Weile die Rechnung zu meiner letzten Bestellung und finde sie in meinem Konto einfach nicht. Wo liegt die.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q062", "query": "Mein Abo laeuft trotz Kuendigung weiter und ich wurde erneut belastet, jetzt will ich die sofortige Kuendigung und die Rueckbuchung der zu viel gezahlten Betraege. Wie setze ich das durch.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q063", "query": "Ihr habt mir eine falsche Rechnung mit zu hohem Betrag geschickt und ich brauche eine korrigierte Rechnung sowie die Rueckbuchung der Differenz. Wie bekomme ich das geregelt.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q064", "query": "Ich zahle seit Monaten fuer zwei Abos, obwohl ich nur eines abgeschlossen habe, und will das zweite gekuendigt und die Doppelzahlungen zurueckgebucht. Wie gehe ich vor.", "category": "Payment & Billing", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    # -- concise (2 simple / 3 ambiguous / 3 clear) --
    {"id": "q065", "query": "Ratenzahlung ueber einen externen Anbieter verfuegbar", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q066", "query": "Studentenrabatt vorhanden ja oder nein", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q067", "query": "Rechnung zur letzten Bestellung herunterladen wo", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q068", "query": "Hinterlegte Kreditkarte aendern wo", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q069", "query": "Gutscheincode im Konto einloesen wie", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q070", "query": "Ratenzahlung fuer teure Bestellung einrichten Ablauf", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q071", "query": "Zahlungsart von Kreditkarte auf SEPA-Lastschrift umstellen Schritte", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q072", "query": "Doppelte Abbuchung reklamieren und Betrag zurueckbuchen lassen wie", "category": "Payment & Billing", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    # ===== Account & Security (24) — Login, 2FA, Datenschutz, Kontoverwaltung =====
    # -- polite (3 simple / 3 ambiguous / 2 clear) --
    {"id": "q073", "query": "Ist es moeglich, sich bei Ihnen kuenftig auch per Apple-ID anzumelden?", "category": "Account & Security", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q074", "query": "Werden meine hinterlegten persoenlichen Daten bei Ihnen verschluesselt gespeichert?", "category": "Account & Security", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q075", "query": "Ich wollte nur kurz anmerken, dass die Zwei-Faktor-Anmeldung bei Ihnen wunderbar reibungslos funktioniert. Sehr beruhigend!", "category": "Account & Security", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q076", "query": "Koennten Sie mir zeigen, wo ich in meinem Konto die Liste der aktiven Sitzungen einsehen kann?", "category": "Account & Security", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q077", "query": "Ich moechte meine hinterlegte E-Mail-Adresse aktualisieren. Wo finde ich diese Einstellung in meinem Konto?", "category": "Account & Security", "sentence_count": 2, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q078", "query": "Wo kann ich einstellen, dass ich bei jeder neuen Anmeldung eine Benachrichtigung per E-Mail erhalte?", "category": "Account & Security", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q079", "query": "Ich habe versehentlich zwei Konten unter verschiedenen E-Mail-Adressen angelegt und moechte diese zu einem einzigen zusammenfuehren. Koennten Sie mir die noetigen Schritte erklaeren?", "category": "Account & Security", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    {"id": "q080", "query": "Mein Vater ist letzten Monat verstorben und hatte ein aktives Konto bei Ihnen. Wie kann ich dieses Konto schliessen und die hinterlegten Daten loeschen lassen?", "category": "Account & Security", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    # -- frustrated (3 simple / 2 ambiguous / 3 clear) --
    {"id": "q081", "query": "Seit dem letzten App-Update muss ich mich bei jedem Start komplett neu einloggen und das nervt inzwischen wirklich.", "category": "Account & Security", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q082", "query": "Ihr habt mein Konto wegen angeblich verdaechtiger Aktivitaet gesperrt, obwohl ich nur im Urlaub in Italien war. So behandelt man keine langjaehrigen Kunden.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q083", "query": "Seit Wochen bekomme ich Spam an genau die Adresse, die ich nur bei euch hinterlegt habe. Ich vermute langsam ein Datenleck bei euch.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q084", "query": "Ich will die Zwei-Faktor-Anmeldung endlich abschalten, finde den Schalter dafuer aber nirgendwo in den Einstellungen. Wo ist der versteckt.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q085", "query": "Ich versuche seit gestern mein Passwort zu aendern und der Button dafuer reagiert einfach nicht. Wo geht das sonst.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q086", "query": "Ich versuche seit gestern den Bestaetigungscode fuer die Zwei-Faktor-Anmeldung zu erhalten, aber die SMS kommt nie an, obwohl mein Empfang einwandfrei ist. Wie richte ich stattdessen eine App als zweiten Faktor ein.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q087", "query": "Ich bin ueberzeugt, dass sich jemand Fremdes in mein Konto eingeloggt hat, und will jetzt sofort alle Sitzungen beenden, das Passwort aendern und die Zwei-Faktor-Anmeldung aktivieren. Wie sichere ich mein Konto ab.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q088", "query": "Ich moechte mein Konto endgueltig loeschen und sichergehen, dass wirklich alle meine Daten entfernt werden. Wie leite ich diese vollstaendige Loeschung ein.", "category": "Account & Security", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    # -- concise (2 simple / 3 ambiguous / 3 clear) --
    {"id": "q089", "query": "Anmeldung mit Fingerabdruck moeglich", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q090", "query": "Passwortlose Anmeldung angeboten", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q091", "query": "Letzte Anmeldungen im Konto einsehen wo", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q092", "query": "Sicherheitsfrage im Konto aendern wo", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q093", "query": "Datenschutzeinstellungen im Konto finden wo", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q094", "query": "Datenauskunft ueber alle gespeicherten Daten anfordern Ablauf", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q095", "query": "Zwei-Faktor-Anmeldung per App komplett neu einrichten Schritte", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q096", "query": "Gesperrtes Konto wieder entsperren lassen Ablauf", "category": "Account & Security", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    # ===== Technical & Other (24) — App/Website, Bewertungen, Newsletter, Empfehlung =====
    # -- polite (3 simple / 3 ambiguous / 2 clear) --
    {"id": "q097", "query": "Ist Ihre App auch mit aelteren Android-Versionen kompatibel?", "category": "Technical & Other", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q098", "query": "Kann ich eine abgegebene Bewertung eigentlich auch anonym veroeffentlichen?", "category": "Technical & Other", "sentence_count": 1, "mood": "polite", "step_warranted": "simple"},
    {"id": "q099", "query": "Ich wollte einfach mal sagen, dass Ihre App wirklich angenehm schnell und uebersichtlich ist. Das nutze ich taeglich gern!", "category": "Technical & Other", "sentence_count": 2, "mood": "polite", "step_warranted": "simple"},
    {"id": "q100", "query": "Koennten Sie mir sagen, wo ich in der App die Sprache auf Englisch umstellen kann?", "category": "Technical & Other", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q101", "query": "Ich moechte die Haeufigkeit Ihrer Newsletter reduzieren, ohne mich ganz abzumelden. Wo finde ich diese Einstellung?", "category": "Technical & Other", "sentence_count": 2, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q102", "query": "Wo finde ich in der App meinen persoenlichen Empfehlungslink, den ich an Freunde weitergeben kann?", "category": "Technical & Other", "sentence_count": 1, "mood": "polite", "step_warranted": "ambiguous"},
    {"id": "q103", "query": "Ich habe vor kurzem ein Hemd gekauft und moechte eine ausfuehrliche Bewertung mit einem eigenen Produktfoto verfassen. Koennten Sie mir erklaeren, wie ich dabei vorgehe?", "category": "Technical & Other", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    {"id": "q104", "query": "Ich moechte mehrere Freunde ueber einen gemeinsamen Empfehlungslink einladen und im Anschluss meine Praemien verwalten. Wie richte ich das ein?", "category": "Technical & Other", "sentence_count": 2, "mood": "polite", "step_warranted": "clear"},
    # -- frustrated (3 simple / 2 ambiguous / 3 clear) --
     {"id": "q105", "query": "Eure staendigen Pop-ups mit Bewertungsaufforderungen nach jedem Einkauf sind wirklich aufdringlich und stoeren mich jedes Mal.", "category": "Technical & Other", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q106", "query": "Ich habe euren Newsletter gezielt fuer Kindermode abonniert und bekomme stattdessen seit Wochen nur Werbung fuer Herrenrasierer.", "category": "Technical & Other", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q107", "query": "Euer Kategoriefilter wirft seit Wochen wild Produkte aus voellig anderen Bereichen dazwischen, und offenbar bekommt ihr das einfach nicht in den Griff.", "category": "Technical & Other", "sentence_count": 1, "mood": "frustrated", "step_warranted": "simple"},
    {"id": "q108", "query": "Ich will meine versehentlich abgegebene Bewertung wieder loeschen, finde dafuer aber keinen Knopf. Wo ist der.", "category": "Technical & Other", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q109", "query": "Ich suche in der App die Option, um den Dark Mode zu aktivieren, und finde sie einfach nicht. Wo versteckt die sich.", "category": "Technical & Other", "sentence_count": 2, "mood": "frustrated", "step_warranted": "ambiguous"},
    {"id": "q110", "query": "Ich habe eine ehrliche Zwei-Sterne-Bewertung abgegeben und der Verkaeufer antwortet darunter mit persoenlichen Beleidigungen. Wie melde ich diesen Kommentar und lasse ihn entfernen.", "category": "Technical & Other", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q111", "query": "Die App stuerzt bei mir seit dem Update jedes Mal beim Abschicken der Bestellung ab und ich komme nicht weiter. Wie bekomme ich das behoben.", "category": "Technical & Other", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    {"id": "q112", "query": "Mein Bruder hat mit meinem Empfehlungslink bestellt, aber die versprochene Praemie ist bei mir nie angekommen. Wie reiche ich das nach und lasse die Gutschrift pruefen.", "category": "Technical & Other", "sentence_count": 2, "mood": "frustrated", "step_warranted": "clear"},
    # -- concise (2 simple / 3 ambiguous / 3 clear) --
    {"id": "q113", "query": "Produktvideos in der App abspielbar ja oder nein", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q114", "query": "Newsletter nur auf Deutsch erhaeltlich", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "simple"},
    {"id": "q115", "query": "Push-Benachrichtigungen der App deaktivieren wo", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q116", "query": "Abgegebene Bewertung nachtraeglich bearbeiten wo", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q117", "query": "Produktfilter zuruecksetzen wo", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "ambiguous"},
    {"id": "q118", "query": "App vollstaendig zuruecksetzen und neu einrichten Schritte", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q119", "query": "Technischen Fehler im Shop melden und dokumentieren wie", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
    {"id": "q120", "query": "Veroeffentlichte Bewertung samt Foto aendern und neu einreichen Ablauf", "category": "Technical & Other", "sentence_count": 1, "mood": "concise", "step_warranted": "clear"},
]

print(f"Queries loaded: {len(test_queries)}")

def print_breakdown(title, values, order=None):
    """Print a count breakdown of one query field, grouped by distinct value.

    Pipeline: called once per field (category, sentence_count, mood,
    step_warranted) right after `test_queries` is defined in Notebook 04,
    to confirm the 120-query evaluation set stays balanced.

    Args:
        title: Heading printed above the breakdown.
        values: Iterable of field values, one per query.
        order: Fixed key display order (e.g. category names in dataset
            order). Falls back to sorted() when omitted, which is correct
            for sentence_count, where sort order and display order match.
    """
    counts = Counter(values)
    keys = order if order else sorted(counts)
    label_width = max(len(str(k)) for k in keys)
    print(f"\n{title}")
    for k in keys:
        print(f"  {str(k):<{label_width}}  {counts[k]:>3}")

category_order = [
    "Orders & Shipping", "Returns & Complaints", "Payment & Billing",
    "Account & Security", "Technical & Other",
]
mood_order = ["polite", "frustrated", "concise"]
step_order = ["simple", "ambiguous", "clear"]

print_breakdown("Category",        (q["category"]        for q in test_queries), category_order)
print_breakdown("Sentence count",  (q["sentence_count"]   for q in test_queries))
print_breakdown("Mood",            (q["mood"]             for q in test_queries), mood_order)
print_breakdown("Step warranted",  (q["step_warranted"]   for q in test_queries), step_order)

Queries loaded: 120

Category
  Orders & Shipping      24
  Returns & Complaints   24
  Payment & Billing      24
  Account & Security     24
  Technical & Other      24

Sentence count
  1   64
  2   55
  3    1

Mood
  polite       40
  frustrated   40
  concise      40

Step warranted
  simple      40
  ambiguous   40
  clear       40


**Generation parameters**

All three models use identical settings so that any differences in output are due to the models themselves, not the sampling configuration.   

`max_new_tokens = 384` is set slightly higher than the 256 used during data generation in Notebook 02.  
The judge in Notebook 05 scores whether a response *ends* with a closing offer to help — a response truncated mid-sentence automatically loses that point regardless of model quality.    
384 tokens gives enough headroom for even verbose Teacher responses to complete naturally, while keeping per-query inference time acceptable on a T4.   

In [5]:
SYSTEM_PROMPT = (
    "Du bist ein professioneller Kundenservice-Mitarbeiter. "
    "Antworte auf Deutsch und halte dich strikt an folgendes Format:\n"
    "Beginne mit einer kurzen Empathie- oder Begruessungsformel (z.B. 'Das tut mir leid', 'Vielen Dank fuer Ihre Anfrage').\n"
    "Gib deine Loesungsschritte als nummerierte Aufzaehlung — nicht als Fliestext.\n"
    "Schliesse mit einem Hilfsangebot ab (z.B. 'Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung').\n"
    "Verwende eine professionelle, hoefliche Sprache (Sie-Form).\n"
    "Maximal 150 Woerter."
)

GEN_PARAMS = {
    "max_new_tokens": 384,
    "temperature":    0.7,
    "top_p":          0.9,
    "do_sample":      True,
}

In [6]:
def run_inference(model, tokenizer, query_text, query_idx):
    """
    Generate one response for a single query.

    Returns:
        response_text       (str)   — decoded model output, prompt stripped
        generated_tokens    (int)   — number of new tokens produced
        generation_time_sec (float) — wall-clock seconds for model.generate()
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": query_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    prompt_len = inputs["input_ids"].shape[1]

    torch.manual_seed(42 + query_idx)

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **GEN_PARAMS,
            pad_token_id=tokenizer.pad_token_id,
        )
    generation_time_sec = time.time() - t0

    generated_ids    = outputs[0][prompt_len:]
    response_text    = tokenizer.decode(generated_ids, skip_special_tokens=True)
    generated_tokens = len(generated_ids)

    return response_text, generated_tokens, generation_time_sec


def reset_vram():
    """
    Hard VRAM reset between model swaps.

    Ensures torch.cuda.max_memory_allocated() starts from zero for each model,
    so peak VRAM measurements are not contaminated by the previous model.
    """
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"VRAM reset — allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Sequential Inference

Runs all three models on the 120 evaluation queries in order: Base → Teacher → Student.

Each model follows the same sequence:
- Hard VRAM reset so peak memory measurements are clean and independent
- Load model
- One warmup generation (discarded) to absorb CUDA kernel compilation overhead
- 120 timed inference calls — responses printed inline
- Peak VRAM recorded
- Model deleted and VRAM reset before the next load

All three models use identical generation parameters (defined in Section 2).

In [7]:
from unsloth import FastLanguageModel
from peft import PeftModel
import warnings
import transformers

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

base_results    = []
teacher_results = []
student_results = []
vram_measurements = []

# Set to False to skip a model — useful when re-running one model
# without waiting for all three to complete again
RUN_BASE    = True
RUN_TEACHER = True
RUN_STUDENT = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 3.1 Base — `Llama-3.2-1B-Instruct` (no fine-tuning)

Baseline: the out-of-the-box 1B model before any training.
Shows what the student model gains from distillation.

In [8]:
if RUN_BASE:
    # --- Load ---
    reset_vram()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Base model ready.\n")

    # --- Warmup ---
    run_inference(model, tokenizer, "Hallo", -1)
    print("Warmup done. Starting inference...\n")

    # --- Inference ---
    for i, entry in enumerate(test_queries):
        response_text, generated_tokens, generation_time_sec = run_inference(
            model, tokenizer, entry["query"], i
        )
        tokens_per_sec = generated_tokens / generation_time_sec
        word_count     = len(response_text.split())

        base_results.append({
            "query_id":            entry["id"],
            "query_category":      entry["category"],
            "query_mood":          entry["mood"],
            "sentence_count":      entry["sentence_count"],
            "query_text":          entry["query"],
            "response_text":       response_text,
            "generated_tokens":    generated_tokens,
            "generation_time_sec": round(generation_time_sec, 3),
            "tokens_per_sec":      round(tokens_per_sec, 1),
            "word_count":          word_count,
        })

        print(f"{'='*60}")
        print(f"[{entry['id']}] {entry['query']}")
        print(f"{'='*60}")
        print(response_text)
        print(f"\n→ {generated_tokens} tok | {generation_time_sec:.1f}s | {tokens_per_sec:.1f} tok/s | {word_count} words\n")

    # --- VRAM ---
    vram_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    vram_measurements.append({"model_label": "base", "vram_peak_gb": round(vram_peak_gb, 2)})
    print(f"Peak VRAM (base): {vram_peak_gb:.2f} GB")

    # --- Unload ---
    del model
    reset_vram()

VRAM reset — allocated: 0.01 GB
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Base model ready.

Warmup done. Starting inference...

[q001] Guten Tag, ich wollte fragen ob Sie auch in die Schweiz liefern. Falls ja, wie hoch waeren die Versandkosten?
Vielen Dank fuer Ihre Anfrage. Hier ist eine kurze Empfehlung:

- Das tut mir leid, dass ich Ihnen keine genaue Antwort auf Ihre Frage geben kann, da ich keine spezifischen Preise für Versandkosten in der Schweiz habe.
- Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung.

→ 77 tok | 1.7s | 46.1 tok/s | 47 words

[q002] Koennten Sie mir sagen, bis zu welcher Uhrzeit ich bestellen muss, damit die Ware noch am selben Tag versendet wird?
Vielen Dank fuer Ihre Anfrage. Bitte bestellt Sie Ihre Ware ab 12:00 Uhr.

→ 22 tok | 0.5s | 42.2 tok/s | 13 words

[q003] Ich wollte einfach mal ein Lob dalassen: meine letzte Lieferung kam blitzschnell und tadellos verpackt an. Machen Sie weiter so!
Vielen Dank für Ihre Lobesworte! Ihre letzte Lieferung war wirklich sehr gut und ich bin froh, dass ich Ihnen die Ware erfolgreich 

### 3.2 Teacher — `LeoLM 7B Chat` + LoRA adapter

The fine-tuned 7B model from Notebook 01.   
Loaded in two steps: the quantised base first, then the LoRA adapter applied on top via PEFT. This is the target style the student was trained to imitate.

In [9]:
if RUN_TEACHER:
    # --- Load ---
    reset_vram()
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=TEACHER_BASE_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(base_model, TEACHER_ADAPTER)
    FastLanguageModel.for_inference(model)
    print("Teacher model ready.\n")

    # --- Warmup ---
    run_inference(model, tokenizer, "Hallo", -1)
    print("Warmup done. Starting inference...\n")

    # --- Inference ---
    for i, entry in enumerate(test_queries):
        response_text, generated_tokens, generation_time_sec = run_inference(
            model, tokenizer, entry["query"], i
        )
        tokens_per_sec = generated_tokens / generation_time_sec
        word_count     = len(response_text.split())

        teacher_results.append({
            "query_id":            entry["id"],
            "query_category":      entry["category"],
            "query_mood":          entry["mood"],
            "sentence_count":      entry["sentence_count"],
            "query_text":          entry["query"],
            "response_text":       response_text,
            "generated_tokens":    generated_tokens,
            "generation_time_sec": round(generation_time_sec, 3),
            "tokens_per_sec":      round(tokens_per_sec, 1),
            "word_count":          word_count,
        })

        print(f"{'='*60}")
        print(f"[{entry['id']}] {entry['query']}")
        print(f"{'='*60}")
        print(response_text)
        print(f"\n→ {generated_tokens} tok | {generation_time_sec:.1f}s | {tokens_per_sec:.1f} tok/s | {word_count} words\n")

    # --- VRAM ---
    vram_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    vram_measurements.append({"model_label": "teacher", "vram_peak_gb": round(vram_peak_gb, 2)})
    print(f"Peak VRAM (teacher): {vram_peak_gb:.2f} GB")

    # --- Unload ---
    del model, base_model
    reset_vram()

VRAM reset — allocated: 0.01 GB
==((====))==  Unsloth 2026.8.22: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Unsloth: LeoLM/leo-mistral-hessianai-7b-chat had a bad pad_token (</s>). Using pad_token = <unk>.


adapter_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  168MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Teacher model ready.

Warmup done. Starting inference...

[q001] Guten Tag, ich wollte fragen ob Sie auch in die Schweiz liefern. Falls ja, wie hoch waeren die Versandkosten?
Guten Tag! Vielen Dank fuer Ihre Anfrage. Die Antwort lautet wie folgt:

1. Empathie- oder Begruessungsformel
2. Versandkosten koennen je nach Produkt und Lieferadresse variieren
3. Gerne senden wir Ihnen eine detaillierte Schätzung
4. Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung

Bitte entschuldigen Sie die Unannehmlichkeiten — bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung. 


→ 142 tok | 9.5s | 14.9 tok/s | 58 words

[q002] Koennten Sie mir sagen, bis zu welcher Uhrzeit ich bestellen muss, damit die Ware noch am selben Tag versendet wird?
1. Das verstehe ich.
2. In den meisten Fällen ist das bis 14 Uhr.
3. Bitte schauen Sie sich die Produktbeschreibung an.
4. Wenn es nicht angegeben ist, ist es in der Regel bis 14 Uhr.
5. Bei weiteren Fragen stehe ich Ihnen gerne zur Verfügung. 


→ 96 tok

### 3.3 Student — `german-support-llama-1b-distilled`

The distilled 1B model from Notebook 03.   
Saved as a merged checkpoint — the LoRA adapter weights were folded into the base weights before upload, so no PEFT loading step is needed.   
Loaded identically to the base model.   

In [10]:
if RUN_STUDENT:
    # --- Load ---
    reset_vram()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=STUDENT_MODEL_ID,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Student model ready.\n")

    # --- Warmup ---
    run_inference(model, tokenizer, "Hallo", -1)
    print("Warmup done. Starting inference...\n")

    # --- Inference ---
    for i, entry in enumerate(test_queries):
        response_text, generated_tokens, generation_time_sec = run_inference(
            model, tokenizer, entry["query"], i
        )
        tokens_per_sec = generated_tokens / generation_time_sec
        word_count     = len(response_text.split())

        student_results.append({
            "query_id":            entry["id"],
            "query_category":      entry["category"],
            "query_mood":          entry["mood"],
            "sentence_count":      entry["sentence_count"],
            "query_text":          entry["query"],
            "response_text":       response_text,
            "generated_tokens":    generated_tokens,
            "generation_time_sec": round(generation_time_sec, 3),
            "tokens_per_sec":      round(tokens_per_sec, 1),
            "word_count":          word_count,
        })

        print(f"{'='*60}")
        print(f"[{entry['id']}] {entry['query']}")
        print(f"{'='*60}")
        print(response_text)
        print(f"\n→ {generated_tokens} tok | {generation_time_sec:.1f}s | {tokens_per_sec:.1f} tok/s | {word_count} words\n")

    # --- VRAM ---
    vram_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    vram_measurements.append({"model_label": "student", "vram_peak_gb": round(vram_peak_gb, 2)})
    print(f"Peak VRAM (student): {vram_peak_gb:.2f} GB")

    # --- Unload ---
    del model
    reset_vram()

VRAM reset — allocated: 0.01 GB
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Student model ready.

Warmup done. Starting inference...

[q001] Guten Tag, ich wollte fragen ob Sie auch in die Schweiz liefern. Falls ja, wie hoch waeren die Versandkosten?
Vielen Dank fuer Ihre Anfrage.

1. Die Lieferung in die Schweiz ist kostenlos.
2. Die Versandkosten variieren je nach Gewicht und Größe des Pakets. Die genaue Kosten berechnen wir gerne in Ihrem Namen.
3. Die Versandkosten in der Schweiz variieren je nach Gewicht und Größe des Pakets. Die genaue Kosten berechnen wir gerne in Ihrem Namen.

Bitte teilen Sie mir Ihre Lieferadresse mit, damit ich sie in die Kosten berechnen kann. Bei weiteren Fragen stehe ich Ihnen gerne zur Verfuegung.

→ 132 tok | 2.9s | 45.8 tok/s | 80 words

[q002] Koennten Sie mir sagen, bis zu welcher Uhrzeit ich bestellen muss, damit die Ware noch am selben Tag versendet wird?
Vielen Dank fuer Ihre Anfrage. Die Ware wird gerne bis zu 3 Tage vor dem bestellten Zeitpunkt versendet. Bitte entschuldigen Sie die Unannehmlichkeiten — bei weiteren Fra

## 4. Save Results

Saves the raw generation results and VRAM measurements as JSON files to the Kaggle working directory.

Switch `if False` to `if True` to activate. The files can then be downloaded from the Kaggle output panel and committed to `results/` in the repository for Notebook 05.

In [11]:
if True:
    RUN_TAG = f"{date.today():%Y%m%d}_{len(test_queries)}q"

    os.makedirs("/kaggle/working/results", exist_ok=True)

    files = {
        f"raw_generations_base_{RUN_TAG}.json":    base_results,
        f"raw_generations_teacher_{RUN_TAG}.json": teacher_results,
        f"raw_generations_student_{RUN_TAG}.json": student_results,
        f"vram_measurements_{RUN_TAG}.json":       vram_measurements,
    }

    for filename, data in files.items():
        path = f"/kaggle/working/results/{filename}"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"Saved: {path}  ({len(data)} entries)")

Saved: /kaggle/working/results/raw_generations_base_20260829_120q.json  (120 entries)
Saved: /kaggle/working/results/raw_generations_teacher_20260829_120q.json  (120 entries)
Saved: /kaggle/working/results/raw_generations_student_20260829_120q.json  (120 entries)
Saved: /kaggle/working/results/vram_measurements_20260829_120q.json  (3 entries)
